# Evaluation

In this notebook, I evaluate my RAG system.

I use a set of 50 questions and run:
*   RAG (with retrieval)
*   Baseline (no retrieval)

The goal is to analyze:
*   Whether the retrieved chunks actually contain the correct information
*   Whether the RAG-generated answers are correct
*   Whether RAG improves performance compared to the baseline




In [2]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from getpass import getpass

In [3]:
os.environ["OPENAI_API_KEY"] = "Your API" #You should replace this part with your own API.

client = OpenAI()

## Load data

I load two files:

*   ds593_chunks_with_embeddings.pkl:

    This file contains the processed course materials, including all text chunks and their precomputed embeddings. It is used for retrieval in the RAG system.

*   evaluation_questions.csv:

    This file contains the list of evaluation 50 questions that will be used to test both the RAG system and the baseline model.

In [18]:
df = pd.read_pickle("ds593_chunks_with_embeddings.pkl")
questions_df = pd.read_csv("evaluation_questions.csv")

questions = questions_df["question"].tolist()

In [19]:
# Print table
print("Number of chunks:", len(df))
print("Number of questions:", len(questions))
df.head()

Number of chunks: 327
Number of questions: 50


,chunk_id,lecture,source_file,page,chunk_index,text,embedding
0,L6_p1_c1,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,1,1,Lecture 6 - Attention Mechanisms\nWelcome back...,"[0.01058197021484375, 0.034942626953125, -0.04..."
1,L6_p2_c1,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,2,1,The classic test\nAgenda for today\n1. Quick r...,"[0.0213775634765625, 0.0101776123046875, -0.01..."
2,L6_p2_c2,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,2,2,From Lecture 5:\nInput sequence -> Encoder -> ...,"[0.0247955322265625, -0.0084075927734375, -0.0..."
3,L6_p2_c3,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,2,3,Long inputs lose information:\nShort sentence ...,"[-0.0220794677734375, 0.01557159423828125, -0...."
4,L6_p3_c1,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,3,1,"Share with your neighbor, can they guess what ...","[0.0221405029296875, -0.01525115966796875, -0...."


## Embedding function

This function converts the 50 evaluation questions into embeddings.

These embeddings are used to match each question with the most relevant chunks.

In [20]:
def get_embedding(text):
    response = client.embeddings.create(model="text-embedding-3-small", input=str(text))
    return response.data[0].embedding

## RAG retrieval function

This function takes each evaluation question and finds the most relevant parts from the course materials.

It compares the question embedding with all chunk embeddings and selects the top k most similar chunks.

In [21]:
def retrieve_chunks(question, top_k=5):
    question_embedding = np.array(get_embedding(question)).reshape(1, -1)
    chunk_embeddings = np.array(df["embedding"].tolist())

    similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]

    results = df.copy()
    results["similarity"] = similarities

    return results.sort_values("similarity", ascending=False).head(top_k)

## Build context

This function takes the top k retrieved chunks and puts them together into one context.

It includes the source and text of each chunk, and this context will be used to help the model answer the question.

In [22]:
def build_context(top_chunks):
    context = ""

    for _, row in top_chunks.iterrows():
        context += f"\n[Source: {row['lecture']}, Page {row['page']}]\n"
        context += str(row["text"])
        context += "\n"

    return context

## RAG answer

This function gets the final answer using RAG.

It finds relevant chunks, builds the context, and then uses the model to answer the question based on that context.

This function is copied from the RAG_pipeline.ipynb and used here for evaluation.

In [23]:
def generate_rag_answer(question, top_k=5):
    top_chunks = retrieve_chunks(question, top_k=top_k)
    context = build_context(top_chunks)

    prompt = f"""
              Use the provided context to answer the question as best as possible.

              If the answer is partially available, explain based on the context.
              Only say "not enough information" if absolutely necessary.

              Context:
              {context}

              Question:
              {question}

              Answer:
              """

    response = client.chat.completions.create(model="gpt-4o",
                                              messages=[{"role": "system", "content": "You answer questions using only the provided course materials."},
                                                        {"role": "user", "content": prompt}], temperature=0)

    return response.choices[0].message.content, top_chunks, context

## Baseline answer

This function uses the baseline method to get the answer.

It does not use retrieval, and directly uses the model to generate the answer based on the question.

This function is used for comparison with the RAG method, so the baseline method does not use RAG.

This function is copied from the baseline.ipynb and used here for evaluation.

In [24]:
def baseline_answer(question):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": "You are a helpful teaching assistant for a Large Language Models course."},
                  {"role": "user", "content": question}], temperature=0)

    return response.choices[0].message.content

## Run evaluation

This part combines the answers generated by RAG and the baseline into an evaluation table, which is used to compare which method performs better.

In [30]:
eval_results = []

for i, q in enumerate(questions, start=1):
    print(f"{i}/{len(questions)}: {q}")

    rag_ans, sources, context = generate_rag_answer(q, top_k=5)
    base_ans = baseline_answer(q)

    eval_results.append({"question_id": i,
                         "question": q,
                         "baseline_answer": base_ans,
                         "rag_answer": rag_ans,
                         "top_text": "\n\n".join(sources["text"].tolist()),
                         "top_similarity": sources.iloc[0]["similarity"],
                         "top_lecture": sources.iloc[0]["lecture"],
                         "top_page": sources.iloc[0]["page"],
                         "retrieved_context": context,
                         "retrieved_sources": " | ".join(
                             sources.apply(lambda row: f"{row['lecture']} p.{row['page']} sim={row['similarity']:.3f}",
                                           axis=1).tolist())})

1/50: What is the bottleneck problem in encoder-decoder models?
2/50: Why do fixed-size context vectors lose information?
3/50: How does attention solve the bottleneck problem?
4/50: What is the intuition behind attention?
5/50: What are Query, Key, and Value in attention?
6/50: How does attention compute similarity between tokens?
7/50: What is the role of softmax in attention?
8/50: Why do we scale by sqrt(d_k) in attention?
9/50: What is the output of the attention mechanism?
10/50: What is self-attention?
11/50: How does self-attention help capture relationships in a sentence?
12/50: What is multi-head attention and why is it useful?
13/50: What are the main components of a transformer?
14/50: What is positional encoding and why is it needed?
15/50: Why does attention not capture word order?
16/50: What is a residual connection?
17/50: Why are residual connections important for training?
18/50: What is layer normalization used for?
19/50: What is a feed-forward network in transform

In [31]:
# Create evaluation table
eval_df = pd.DataFrame(eval_results)
eval_df.head()

,question_id,question,baseline_answer,rag_answer,top_text,top_similarity,top_lecture,top_page,retrieved_context,retrieved_sources
0,1,What is the bottleneck problem in encoder-deco...,The bottleneck problem in encoder-decoder mode...,The bottleneck problem in encoder-decoder mode...,From Lecture 5:\nInput sequence -> Encoder -> ...,0.596855,Lecture 6 - Attention Mechanisms,2,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.2 sim=0.597...
1,2,Why do fixed-size context vectors lose informa...,"Fixed-size context vectors, often used in trad...",Fixed-size context vectors lose information be...,What we learned today\nAttention solves the bo...,0.546057,Lecture 6 - Attention Mechanisms,18,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.18 sim=0.54...
2,3,How does attention solve the bottleneck problem?,The bottleneck problem in sequence processing ...,Attention solves the bottleneck problem by usi...,What we learned today\nAttention solves the bo...,0.647113,Lecture 6 - Attention Mechanisms,18,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.18 sim=0.64...
3,4,What is the intuition behind attention?,The intuition behind attention in the context ...,The intuition behind attention is to allow a m...,This is exactly how attention works!\nAttentio...,0.624338,Lecture 6 - Attention Mechanisms,4,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.4 sim=0.624...
4,5,"What are Query, Key, and Value in attention?","In the context of attention mechanisms, partic...","In attention mechanisms, Query (Q), Key (K), a...",Three roles in attention\nAttention uses three...,0.705792,Lecture 6 - Attention Mechanisms,3,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.3 sim=0.706...


## Add manual scoring columns

This part adds several columns to the evaluation table for manual evaluation.

These columns are used for humans to check whether the retrieved chunks contain the answer we need, and to compare whether the RAG answer or the baseline answer is better.

In [32]:
eval_df["retrieved_contains_answer"] = ""
eval_df["rag_correct"] = ""
eval_df["baseline_correct"] = ""
eval_df["notes"] = ""

eval_df.head()

,question_id,question,baseline_answer,rag_answer,top_text,top_similarity,top_lecture,top_page,retrieved_context,retrieved_sources,retrieved_contains_answer,rag_correct,baseline_correct,notes
0,1,What is the bottleneck problem in encoder-deco...,The bottleneck problem in encoder-decoder mode...,The bottleneck problem in encoder-decoder mode...,From Lecture 5:\nInput sequence -> Encoder -> ...,0.596855,Lecture 6 - Attention Mechanisms,2,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.2 sim=0.597...,,,,
1,2,Why do fixed-size context vectors lose informa...,"Fixed-size context vectors, often used in trad...",Fixed-size context vectors lose information be...,What we learned today\nAttention solves the bo...,0.546057,Lecture 6 - Attention Mechanisms,18,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.18 sim=0.54...,,,,
2,3,How does attention solve the bottleneck problem?,The bottleneck problem in sequence processing ...,Attention solves the bottleneck problem by usi...,What we learned today\nAttention solves the bo...,0.647113,Lecture 6 - Attention Mechanisms,18,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.18 sim=0.64...,,,,
3,4,What is the intuition behind attention?,The intuition behind attention in the context ...,The intuition behind attention is to allow a m...,This is exactly how attention works!\nAttentio...,0.624338,Lecture 6 - Attention Mechanisms,4,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.4 sim=0.624...,,,,
4,5,"What are Query, Key, and Value in attention?","In the context of attention mechanisms, partic...","In attention mechanisms, Query (Q), Key (K), a...",Three roles in attention\nAttention uses three...,0.705792,Lecture 6 - Attention Mechanisms,3,"\n[Source: Lecture 6 - Attention Mechanisms, P...",Lecture 6 - Attention Mechanisms p.3 sim=0.706...,,,,


In [33]:
# Create CVS file
eval_df.to_csv("evaluation_results.csv", index=False)

## Conclusion

In this project, I generated a dataset based on the course lecture materials and built a RAG system. I also compared the RAG system with a baseline model that does not use retrieval. The final evaluation results show that the RAG method performs better overall.

In most cases, the retrieved text chunks do contain relevant information, which helps the model generate more accurate answers. Although the similarity scores are not very high (around 0.43 to 0.78), the retrieved content is still helpful for answering the questions. This shows that semantic similarity does not need to be very high to be effective.

Compared to the baseline model, the answers generated by RAG are usually more detailed and closer to the actual lecture content. Also, the answers from RAG are indeed shorter than those from the baseline model and can highlight the key points more clearly. In contrast, although the baseline model can sometimes generate correct answers, they are often more general, less organized, and may miss important details. The answers generated by the baseline model make it harder for students to understand the key concepts, because the baseline answers may include unnecessary or less important information.

So, this evaluation shows that adding retrieval can improve the quality of the answers and make the model more reliable, especially when working with specific materials such as course lectures.